# Covariance Protocol Histories Characterization Report

This local CPU notebook loads the derived outputs produced by
`scripts/analyze_covariance_protocol_histories_characterization.py`
and renders the same report figures inline, one section at a time.

## Source Campaign And Derived Outputs

Default source pointer:
`colab_small_system_testing/gpu_data/covariance_protocol_histories/latest_campaign.json`

Default analysis output root:
`colab_small_system_testing/analysis_outputs/covariance_protocol_histories_characterization_cpu/<campaign_id>/`

The notebook does not recompute metrics. It only reads the saved CSV/NPZ/JSON products and rebuilds the same figures cell by cell.

## 1. Setup

Locate the repo root, import the analysis module, and configure plotting defaults for inline display.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise FileNotFoundError('Could not locate repo root with src/')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scripts.analyze_covariance_protocol_histories_characterization as covchar

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


## 2. Load Derived Characterization Outputs

Resolve the latest covariance-history campaign, locate the matching derived output directory, and load the saved tables and manifest.

In [ ]:
campaign_pointer = covchar.DEFAULT_CAMPAIGN_POINTER
campaign_manifest, campaign_manifest_path, gpu_data_root = covchar.load_campaign(campaign_pointer)
campaign_id = str(campaign_manifest['campaign_id'])
output_root = covchar.default_analysis_output_root(campaign_id)
payload = covchar.load_analysis_outputs(output_root)

analysis_manifest = payload['manifest']
metrics_df = payload['metrics_df']
frob_df = payload['frob_df']
ensemble_df = payload['ensemble_df']

print(f'[campaign] {campaign_id}')
print(f'[campaign manifest] {campaign_manifest_path}')
print(f'[analysis output root] {output_root}')
print(f'[metrics rows] {len(metrics_df)}')
print(f'[frob rows] {len(frob_df)}')
print(f'[ensemble summary rows] {len(ensemble_df)}')
print(f'[pdf] {analysis_manifest['files'].get('pdf', 'not generated')}')

display(pd.DataFrame(analysis_manifest['run_configs']))


## 3. One Overview Page Per System Size

These three figures reproduce the first three PDF pages. Each page uses a `4 x 2` layout with protocols in columns and the following rows:

1. Frobenius successive-cycle deltas
2. Late-ensemble entropy-slope histograms
3. Late-ensemble real-space Chern histograms
4. Late-ensemble normalized-trace histograms

In [ ]:
for nx, ny in covchar.GEOMETRY_ORDER:
    geometry_key = f'N{nx}x{ny}'
    display(Markdown(f'### Overview: `{geometry_key}`'))
    fig = covchar.create_overview_figure(geometry_key, metrics_df, frob_df)
    display(fig)
    plt.close(fig)


## 4. `N16x20` Entropy-Slope Snapshot Distributions

These two figures reproduce the cycle-snapshot slope comparison pages for cycles `5, 6, 7, 8, 9, 10`.

In [ ]:
for protocol in covchar.PROTOCOL_ORDER:
    display(Markdown(f'### `N16x20` `{protocol}` slope snapshots'))
    fig = covchar.create_snapshot_histogram_figure(
        metrics_df,
        geometry_key='N16x20',
        protocol=protocol,
        metric_col='entropy_slope',
        metric_label='entropy slope',
    )
    display(fig)
    plt.close(fig)


## 5. `N16x20` Real-Space Chern Snapshot Distributions

These two figures reproduce the cycle-snapshot Chern comparison pages for cycles `5, 6, 7, 8, 9, 10`.

In [ ]:
for protocol in covchar.PROTOCOL_ORDER:
    display(Markdown(f'### `N16x20` `{protocol}` Chern snapshots'))
    fig = covchar.create_snapshot_histogram_figure(
        metrics_df,
        geometry_key='N16x20',
        protocol=protocol,
        metric_col='real_space_chern',
        metric_label='real-space Chern number',
    )
    display(fig)
    plt.close(fig)


## 6. `N16x20` Normalized-Trace Snapshot Distributions

These two figures reproduce the cycle-snapshot normalized-trace comparison pages for cycles `5, 6, 7, 8, 9, 10`.

In [ ]:
for protocol in covchar.PROTOCOL_ORDER:
    display(Markdown(f'### `N16x20` `{protocol}` normalized-trace snapshots'))
    fig = covchar.create_snapshot_histogram_figure(
        metrics_df,
        geometry_key='N16x20',
        protocol=protocol,
        metric_col='normalized_trace',
        metric_label='tr(G) / Nlayer',
    )
    display(fig)
    plt.close(fig)


## 7. Late-Ensemble Entropy-Slope Comparisons Across System Size

These figures reproduce the late-ensemble slope histogram pages for `cycles 6-10`, one page per protocol.

In [ ]:
for protocol in covchar.PROTOCOL_ORDER:
    display(Markdown(f'### Late-ensemble entropy slopes: `{protocol}`'))
    fig = covchar.create_late_ensemble_compare_figure(
        metrics_df,
        protocol=protocol,
        metric_col='entropy_slope',
        metric_label='entropy slope',
    )
    display(fig)
    plt.close(fig)


## 8. Late-Ensemble Real-Space Chern Comparisons Across System Size

These figures reproduce the late-ensemble Chern histogram pages for `cycles 6-10`, one page per protocol.

In [ ]:
for protocol in covchar.PROTOCOL_ORDER:
    display(Markdown(f'### Late-ensemble real-space Chern: `{protocol}`'))
    fig = covchar.create_late_ensemble_compare_figure(
        metrics_df,
        protocol=protocol,
        metric_col='real_space_chern',
        metric_label='real-space Chern number',
    )
    display(fig)
    plt.close(fig)


## 9. Late-Ensemble Normalized-Trace Comparisons Across System Size

These figures reproduce the late-ensemble normalized-trace histogram pages for `cycles 6-10`, one page per protocol.

In [ ]:
for protocol in covchar.PROTOCOL_ORDER:
    display(Markdown(f'### Late-ensemble normalized trace: `{protocol}`'))
    fig = covchar.create_late_ensemble_compare_figure(
        metrics_df,
        protocol=protocol,
        metric_col='normalized_trace',
        metric_label='tr(G) / Nlayer',
    )
    display(fig)
    plt.close(fig)
